# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/swiftcoder0/flyrank_intership_assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Install required packages
%pip install -q duckdb huggingface_hub pandas

from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(HF_TOKEN)

print("✅ Hugging Face login successful")

✅ Hugging Face login successful


In [2]:
import duckdb

con = duckdb.connect()

print("DuckDB version:", duckdb.__version__)
print("✅ DuckDB is ready")

DuckDB version: 1.3.2
✅ DuckDB is ready


In [3]:
import duckdb

con = duckdb.connect()

con.sql("""
INSTALL httpfs;
LOAD httpfs;
""")

print("✅ DuckDB HTTPFS loaded")

✅ DuckDB HTTPFS loaded


## 1. Unit of analysis + time window

**Unit of analysis:** One content page (identified by `content_hash_id`) for a specific client. Each row represents the performance of one content item over the selected analysis period.

**Time window:** I will use the mid-panel month **2026-03** because it avoids the final outcome window and follows the assignment recommendation.

**Purpose:** My lane is **Refresh / Content Opportunity Scoring**. The goal is to identify which content pages should be prioritized for review or refresh using observable search performance signals.

In [4]:
import duckdb

con = duckdb.connect()

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_pages,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {fact_daily}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_pages,start_date,end_date
0,9841378,331437,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

### Features
- impressions
- clicks
- average position
- CTR
- days since last update

These features are observable before making a refresh decision and can help prioritize pages.

### Label / Proxy
The target is whether a page shows declining performance and should be prioritized for refresh.

### Context
- client_hash_id
- content_hash_id
- report_date

These identify the content and provide context but are not predictive features.

### Excluded
- trend_direction
- trend_pct

Reason: These are derived from the outcome and would cause data leakage because they directly reveal the target.

In [6]:
# Query 1 - Row count

con.sql(f"""
SELECT COUNT(*) AS rows
FROM {fact_daily}
""").df()

# Query 2 - Date window

con.sql(f"""
SELECT
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM {fact_daily}
""").df()

# Query 3 - Availability check

con.sql(f"""
SELECT COUNT(*) AS available_rows
FROM {fact_daily}
WHERE gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


## 3. Verify it with queries (grain, counts, missing values, windows)

The verification queries confirm that my selected data comes from the **2026-03** partition. The results show the row count, date range, and the number of rows where `gsc_data_available` is `TRUE`. These checks verify that the data matches the intended unit of analysis and is suitable for feature engineering for my lane.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

This dataset can support decision-making for content refresh prioritization, but it cannot prove why a page's performance changed. External factors such as search engine algorithm updates, seasonality, competition, or content quality are not fully captured. The dataset also represents historical observations, so the findings should be treated as decision-support rather than causal proof.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card.